# 1. Import Library
Mengimpor library yang dibutuhkan untuk pengolahan data, waktu, dan visualisasi.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# 2. Load Dataset
Memuat data jadwal pertandingan IPL dari file CSV dan menampilkan beberapa baris pertama serta informasi dataset.

In [ ]:
# Memuat dataset
df = pd.read_csv('ipl_schedule.csv')

# Menampilkan 5 data teratas
print("5 Data Teratas:")
display(df.head())

# Menampilkan informasi dataset
print("\nInfo Dataset:")
df.info()

# 3. Data Preprocessing
Membersihkan dan menyiapkan data, termasuk mengubah format waktu menjadi tipe datetime, menggabungkan tanggal dan waktu, serta mengurutkan data.

In [ ]:
# Membuat salinan dataframe untuk diproses
df_processed = df.copy()

# Memeriksa missing values
print("Missing values sebelum preprocessing:")
print(df_processed.isnull().sum())

# Menggabungkan DATE, TIME, dan PM/AM menjadi satu kolom string
df_processed['DATETIME_STR'] = df_processed['DATE'] + ' ' + df_processed['TIME'] + ' ' + df_processed['PM/AM']

# Mengubah string menjadi tipe datetime
# Menggunakan dayfirst=True karena sebagian besar format adalah DD/MM/YY
df_processed['DATETIME'] = pd.to_datetime(df_processed['DATETIME_STR'], dayfirst=True)

# Menambahkan durasi pertandingan (asumsi 3.5 jam untuk satu match T20)
match_duration = timedelta(hours=3, minutes=30)
df_processed['START_TIME'] = df_processed['DATETIME']
df_processed['END_TIME'] = df_processed['START_TIME'] + match_duration

# Mengurutkan data berdasarkan waktu pertandingan
df_processed = df_processed.sort_values(by='START_TIME').reset_index(drop=True)

# Menghapus kolom yang sudah tidak diperlukan
df_processed = df_processed.drop(columns=['DATE', 'TIME', 'PM/AM', 'DATETIME_STR', 'DATETIME'])

print("\nData setelah preprocessing:")
display(df_processed.head())

# 4. Definisi Masalah

**Tujuan:** Meminimalkan konflik waktu pertandingan dalam jadwal IPL.

**Konflik terjadi jika:**
1. Dua pertandingan diadakan di **venue (stadion) yang sama** pada rentang waktu yang bertabrakan.
2. Sebuah **tim bermain di waktu yang bertabrakan** (bermain di dua tempat sekaligus, atau waktu pertandingan tumpang tindih).

**Pendekatan:**
Kita akan menyelesaikan masalah ini menggunakan **Algoritma Greedy**. Algoritma ini akan memilih jadwal secara berurutan berdasarkan waktu mulai, dan hanya memasukkan jadwal tersebut jika tidak menimbulkan konflik dengan jadwal yang sudah terpilih.

# 5. Implementasi Greedy Algorithm
Membangun fungsi `greedy_schedule()` untuk menyeleksi pertandingan.

*Catatan: Pada kasus Interval Scheduling standar, algoritma Greedy yang paling optimal adalah dengan memilih berdasarkan waktu selesai paling awal (Earliest Finish Time First). Karena durasi setiap pertandingan di sini dianggap sama (3.5 jam), pengurutan berdasarkan Start Time atau End Time akan memberikan hasil yang ekuivalen.*

In [ ]:
def check_conflict(match, selected_matches):
    """
    Mengecek apakah suatu pertandingan memiliki konflik dengan pertandingan yang sudah terpilih.
    """
    for selected in selected_matches:
        # Cek apakah waktu pertandingan bertabrakan
        # Tabrakan waktu terjadi jika StartA < EndB DAN StartB < EndA
        time_overlap = (match['START_TIME'] < selected['END_TIME']) and (selected['START_TIME'] < match['END_TIME'])
        
        if time_overlap:
            # 1. Konflik Venue: Jika venue sama di waktu yang bertabrakan
            if match['VENUE'] == selected['VENUE']:
                return True
            
            # 2. Konflik Tim: Jika ada tim yang sama bermain di waktu yang bertabrakan
            teams_in_match = [match['HOME TEAM'], match['AWAY TEAM']]
            teams_in_selected = [selected['HOME TEAM'], selected['AWAY TEAM']]
            
            # Cek irisan tim (apakah ada tim yang sama)
            if len(set(teams_in_match).intersection(set(teams_in_selected))) > 0:
                return True
            
    return False

def greedy_schedule(df):
    """
    Fungsi utama algoritma Greedy untuk optimasi penjadwalan.
    """
    # 1. Urutkan pertandingan berdasarkan waktu
    # Data sudah diurutkan saat preprocessing, namun kita pastikan lagi
    sorted_df = df.sort_values(by='START_TIME').reset_index(drop=True)
    
    selected_matches = []
    rejected_matches = []
    
    # 2. Iterasi satu per satu
    for index, row in sorted_df.iterrows():
        # 3. Pilih pertandingan jika tidak ada konflik
        if not check_conflict(row, selected_matches):
            selected_matches.append(row)
        else:
            rejected_matches.append(row)
            
    # 4. Simpan hasil jadwal optimal
    df_selected = pd.DataFrame(selected_matches)
    df_rejected = pd.DataFrame(rejected_matches)
    
    return df_selected, df_rejected

# 6. Eksekusi Algoritma
Menjalankan fungsi `greedy_schedule()` pada data yang sudah diproses dan menampilkan hasilnya.

In [ ]:
# Menjalankan algoritma greedy
optimal_schedule, rejected_schedule = greedy_schedule(df_processed)

# Menampilkan hasil
print(f"Total jadwal awal: {len(df_processed)}")
print(f"Total jadwal terpilih (Optimal): {len(optimal_schedule)}")
print(f"Total jadwal ditolak (Konflik): {len(rejected_schedule)}")

print("\n--- 5 Jadwal Optimal Pertama ---")
display(optimal_schedule[['HOME TEAM', 'AWAY TEAM', 'VENUE', 'START_TIME', 'END_TIME']].head())

# 7. Evaluasi Hasil
Menghitung persentase pertandingan yang berhasil dijadwalkan tanpa konflik dan jumlah konflik yang dihindari.

In [ ]:
total_initial = len(df_processed)
total_selected = len(optimal_schedule)
total_conflicts_avoided = len(rejected_schedule)

selection_rate = (total_selected / total_initial) * 100
conflict_rate = (total_conflicts_avoided / total_initial) * 100

print("=== LAPORAN EVALUASI PENJADWALAN ===")
print(f"Total Data Pertandingan Awal : {total_initial} pertandingan")
print(f"Pertandingan Terpilih        : {total_selected} pertandingan ({selection_rate:.2f}%)")
print(f"Konflik Berhasil Dihindari   : {total_conflicts_avoided} pertandingan ({conflict_rate:.2f}%)")

# 8. Visualisasi
Membandingkan jumlah jadwal sebelum dan sesudah optimasi dengan algoritma Greedy.

In [ ]:
# Menyiapkan data untuk visualisasi
categories = ['Total Jadwal Awal', 'Jadwal Terpilih\n(Optimal)', 'Jadwal Ditolak\n(Konflik)']
values = [total_initial, total_selected, total_conflicts_avoided]
colors = ['#3498db', '#2ecc71', '#e74c3c']

plt.figure(figsize=(10, 6))
bars = plt.bar(categories, values, color=colors)

# Menambahkan label nilai di atas bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, int(yval), ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.title('Perbandingan Jumlah Pertandingan Sebelum dan Sesudah Optimasi', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Jumlah Pertandingan', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, max(values) + int(max(values)*0.15)) # Kasih ruang sedikit di atas

plt.tight_layout()
plt.show()

# 9. Kesimpulan

**Hasil Implementasi:**
Algoritma Greedy berhasil menyeleksi jadwal dan mengurangi konflik. Pertandingan yang menyebabkan tabrakan venue atau memaksa sebuah tim bermain di dua tempat sekaligus pada waktu yang bersamaan telah berhasil diidentifikasi dan dipisahkan.

**Kelebihan Algoritma Greedy pada Kasus Ini:**
1. **Sederhana dan Cepat:** Algoritma berjalan sangat cepat (kompleksitas waktu rendah) karena hanya melakukan satu iterasi linear $\mathcal{O}(N)$ setelah proses pengurutan $\mathcal{O}(N \log N)$.
2. **Mudah Diimplementasikan:** Logika pemilihannya intuitif dan sangat mudah disesuaikan jika ingin menambahkan *rule* atau kendala baru.
3. **Optimal untuk Durasi Tetap:** Pada kasus *Interval Scheduling*, algoritma Greedy yang memilih jadwal berdasarkan waktu selesai paling awal terbukti memberikan solusi global optimal. Karena durasi pertandingan diasumsikan sama (3.5 jam), pengurutan berdasarkan *start time* sudah memenuhi syarat keoptimalan tersebut.

**Kekurangan Algoritma Greedy:**
1. **Tidak Menjadwalkan Ulang (*Rescheduling*):** Algoritma ini murni hanya menyeleksi (*filter*) jadwal yang bentrok dan menolaknya. Pada aplikasi nyata, kita mungkin ingin menggeser pertandingan yang bentrok ke jam/hari lain, bukan membuangnya.
2. **Hanya Mengandalkan Local Optimum:** Greedy tidak bisa mengubah keputusan masa lalu yang mungkin akan memunculkan konfigurasi keseluruhan yang lebih baik jika ditinjau menggunakan kombinatorika lebih kompleks.